In [120]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import pprint

In [121]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


## Configuration

In [122]:
D_MODEL = 128            # Embedding size
BLOCK_SIZE = 64

## Dataset

In [123]:
data_path = Path("../../datasets/tinyshakespeare.txt")
text = data_path.read_text(encoding="utf-8")
chars = sorted(set(text))
VOCAB_SIZE = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)



data = torch.tensor(encode(text), dtype=torch.long)

print("Characters:", len(text))
print(pprint.pprint(stoi, compact=True))
print(f"VOCAB_SIZE = {VOCAB_SIZE}")
print("Encoded shape:", data)    

Characters: 1115393
{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}
None
VOCAB_SIZE = 65
Encoded shape: tensor([18, 47, 56,  ..., 52, 45,  8])


## Tokenizer

## Simple model

In [124]:
model = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, D_MODEL),
    nn.Linear(D_MODEL, VOCAB_SIZE)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

## Training

In [134]:
model = model.to(DEVICE)

running = []
for step in range(10000):
    
    i = torch.randint(0, len(data) - BLOCK_SIZE -1 , (1,)).item()
    
    x = data[i:i+block_size].to(DEVICE)
    y = data[i+1:i+block_size+1].to(DEVICE)

    # print("x:", decode(x.tolist()))
    # print("y:", decode(y.tolist()))


    logits = model(x)

    loss = F.cross_entropy( logits, y )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running.append(loss.item())
    if step % 100 == 0:
        print(step, sum(running[-100:]) / len(running[-100:]))
    

0 2.4122090339660645
100 2.519057521820068
200 2.488853678703308
300 2.5074604463577272
400 2.539639472961426
500 2.4794111180305483
600 2.49952894449234
700 2.531887301206589
800 2.475860185623169
900 2.510559859275818
1000 2.4843422389030456
1100 2.5316862630844117
1200 2.5312423586845396
1300 2.49947336435318
1400 2.5221269953250887
1500 2.4545958137512205
1600 2.527151918411255
1700 2.506914126873016
1800 2.4611811733245847
1900 2.5004869890213013
2000 2.476174967288971
2100 2.5181372213363646
2200 2.501368179321289
2300 2.523853681087494
2400 2.566669406890869
2500 2.490521785020828
2600 2.4812725234031676
2700 2.503169138431549
2800 2.457114543914795
2900 2.5132450330257416
3000 2.486891511678696
3100 2.515838770866394
3200 2.5137846803665163
3300 2.452040148973465
3400 2.49361319065094
3500 2.5127095198631286
3600 2.5061530828475953
3700 2.506491937637329
3800 2.4910463881492615
3900 2.517253563404083
4000 2.471868611574173
4100 2.506628625392914
4200 2.507644749879837
4300 2.50

In [135]:
# -------------------------
# Generate
# -------------------------

x = torch.tensor(
    [stoi["A"]],
    device=DEVICE
)

model.eval()

with torch.no_grad():

    for _ in range(500):

        # x[-1:] has shape [1]
        logits = model(x[-1:])

        # If logits is [1, vocab_size],
        # take the last prediction
        logits = logits[-1]

        probs = torch.softmax(
            logits,
            dim=-1
        )

        # multinomial gives [1]
        next_token = torch.multinomial(
            probs,
            1
        )

        # Both are now [1]
        x = torch.cat([
            x,
            next_token
        ])

print("\nGenerated:\n")
print("".join(itos[i.item()] for i in x))


Generated:

ANICHen adothe au mu y now!
D as m.
SAUPens; by p fer m wis e be Whes.
TA:

O:
BENofer, itht, this iord IUCAUCHasn!ovelds,
Y moncod;
DUNong.
Nwishid peron:
Set ns d,
NELO:

DIN:
Ther?
I haugner ar do ar, y ha aicous t my V:
ISoupar,
Cifo is u, festl ckizere Latecozen revo ifones his bourither htr, winkerur int blils iowu hate.
Butrinods?
Beveawneen wiusitusut woth wseryotlin t s m he trves llllllar asthe sthe llone?
F rer w dwe he
OFoshu hry lve ape we comou tt moifrce
LAn hate he at navitse:
Th!
